In [0]:
from pyspark.sql import Row

# Create small sample data
data = [
    Row(id=1, name="Alice",   amount=100),
    Row(id=2, name="Bob",     amount=250),
    Row(id=3, name="Carol",   amount=75),
    Row(id=4, name="David",   amount=500),
    Row(id=5, name="Eve",     amount=320),
]

df = spark.createDataFrame(data)

# Write to a Delta table (overwrite to keep it idempotent)
df.write.mode("overwrite").saveAsTable("workspace.default.job_demo_data")

print(f"Wrote {df.count()} rows to workspace.default.job_demo_data")

# Pass Data Between Tasks (Task Values)

Without task values, tasks are isolated — Task 2 can't know anything about Task 1's results.

| Scenario | Without Task Values | With Task Values |
|---|---|---|
| Task 1 finds 0 records | Task 2 runs anyway, wastes time | Task 2 checks value → skips if 0 |
| Task 1 discovers a file path | Task 2 has to hard-code the path | Task 2 reads the dynamic path |
| Task 1 needs to pass a count | No way to pass it | Simple `set()` and `get()` |
| Task 1 builds a JSON config | Task 2 can't access it | Task 2 reads the JSON |


In [0]:
row_count = df.count()
print(f" Wrote {row_count} rows")

# Set the task value — Task 2 will read this
dbutils.jobs.taskValues.set(key="row_count", value=row_count)

# Also pass a status message
dbutils.jobs.taskValues.set(key="status", value="SUCCESS")

print(f"Task value 'row_count' set to {row_count}")
print(f"Task value 'status' set to SUCCESS")

In [0]:
# Save the notebook. (Don't run it manually yet — running it standalone will fail because dbutils.jobs.taskValues only works inside a job.)